In [ ]:
import numpy as np
import scipy.stats as si
import sympy as sy
from sympy.stats import Normal, cdf

from scipy.stats import multivariate_normal as mvn
from scipy.optimize import minimize

In [ ]:
S, X2, r, sigma = 200, 250, 0.05, 0.5

T2=3
T1=2
t=0
X1=20

In [ ]:
def e_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r +0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = (np.log(S / K) + (r -  0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    call =(S * si.norm.cdf(d1, 0.0, 1.0) - K * np.exp(-r * T) * si.norm.cdf(d2, 0.0, 1.0))
    return call

In [20]:
def find_critical_stock_price(S, X1, X2, T1, T2, r, sigma):
    tau = T2 - T1

    bounds = [(0, None)]

    objective = lambda S_array: e_call(S_array[0], X2, tau, r, sigma)
    apply_constraint1 = lambda S_array: e_call(S_array[0], X2, tau, r, sigma) - X1

    my_constraints = ({
        "type": "eq",
        "fun": apply_constraint1
    })

    result = minimize(
        objective,
        [S],
        bounds=bounds,
        constraints=my_constraints,
        method="SLSQP"
    )

    return result.x[0]

In [21]:
def call_on_call(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = find_critical_stock_price(S, X1, X2, T1, T2, r, sigma)

    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2) * (T1 - t)) / (
        sigma * np.sqrt(T1 - t)
    )
    D2 = D1 - sigma * np.sqrt(T1 - t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2) * (T2 - t)) / (
        sigma * np.sqrt(T2 - t)
    )
    E2 = E1 - sigma * np.sqrt(T2 - t)

    corr = np.sqrt(T1 / T2)

    dist = mvn(
        mean=np.array([0, 0]),
        cov=np.array([
            [1, corr],
            [corr, 1]
        ])
    )

    N2D1E1 = dist.cdf(np.array([D1, E1]))
    N2D2E2 = dist.cdf(np.array([D2, E2]))
    ND2 = si.norm.cdf(D2, 0.0, 1.0)

    call_call = (
        S * N2D1E1
        - X2 * np.exp(-r * (T2 - t)) * N2D2E2
        - X1 * np.exp(-r * (T1 - t)) * ND2
    )

    return call_call


In [22]:
call_call = call_on_call(S, X1, X2, T1, T2, t, r, sigma)

call_call

np.float64(51.53481014035137)

## Call_put

In [23]:
def e_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (
        sigma * np.sqrt(T)
    )
    d2 = d1 - sigma * np.sqrt(T)

    put = (
        K * np.exp(-r * T) * si.norm.cdf(-d2, 0.0, 1.0)
        - S * si.norm.cdf(-d1, 0.0, 1.0)
    )

    return put

In [24]:
def find_critical_stock_price_call_put(S, X1, X2, T1, T2, r, sigma):
    tau = T2 - T1

    bounds = [(0, None)]

    objective = lambda S_array: e_put(S_array[0], X2, tau, r, sigma)
    apply_constraint1 = lambda S_array: e_put(S_array[0], X2, tau, r, sigma) - X1

    my_constraints = ({
        "type": "eq",
        "fun": apply_constraint1
    })

    result = minimize(
        objective,
        [S],
        bounds=bounds,
        constraints=my_constraints,
        method="SLSQP"
    )

    return result.x[0]


In [25]:
def call_on_put(S, X1, X2, T1, T2, t, r, sigma):
    SOpt = find_critical_stock_price_call_put(S, X1, X2, T1, T2, r, sigma)

    D1 = (np.log(S / SOpt) + (r + 0.5 * sigma ** 2) * (T1 - t)) / (
        sigma * np.sqrt(T1 - t)
    )
    D2 = D1 - sigma * np.sqrt(T1 - t)

    E1 = (np.log(S / X2) + (r + 0.5 * sigma ** 2) * (T2 - t)) / (
        sigma * np.sqrt(T2 - t)
    )
    E2 = E1 - sigma * np.sqrt(T2 - t)

    corr = np.sqrt((T1 - t) / (T2 - t))

    dist = mvn(
        mean=np.array([0, 0]),
        cov=np.array([
            [1, corr],
            [corr, 1]
        ])
    )

    N2_minus_D1_minus_E1 = dist.cdf(np.array([-D1, -E1]))
    N2_minus_D2_minus_E2 = dist.cdf(np.array([-D2, -E2]))
    N_minus_D2 = si.norm.cdf(-D2, 0.0, 1.0)

    call_put = (
        X2 * np.exp(-r * (T2 - t)) * N2_minus_D2_minus_E2
        - S * N2_minus_D1_minus_E1
        - X1 * np.exp(-r * (T1 - t)) * N_minus_D2
    )

    return call_put


In [ ]:
call_put = call_on_put(S, X1, X2, T1, T2, t, r, sigma)

call_put

np.float64(61.08435109518262)